# 2. Extract MetacellFormer embeddings from individual human PBMC cells

Run from the repository root with the **MetacellFormer-env** Python 3.11 kernel. You can start here to use the pretrained model; MetaQ and notebook 01 are not needed. Each row of the downstream input is one single cell, even though the model was pretrained using metacells.

Download only the human PBMC_368K input, its roster, and the five model files:

If access is restricted, run `hf auth login` once before downloading.

```bash
python -m pip install huggingface_hub
hf download netbiolab/MetacellFormer-evaluation inputs/cell_annotation/PBMC_368K.h5ad contracts/cell_type_annotation/PBMC_368K_roster.json --repo-type dataset --local-dir ../MetacellFormer-evaluation
hf download netbiolab/MetacellFormer config.json model.safetensors gene_vocabulary.json gene_reference_medians.npy training_resources/value_bin_edges.npy --local-dir ../MetacellFormer-model
```

These paths match the defaults below; set `METACELLFORMER_EVALUATION` and `METACELLFORMER_MODEL` for other locations. The released input contains 4,638 cells. This notebook uses the first six per class and split in roster order: 96 cells across eight types. **CPU** works (about 2 minutes on the tested two-thread CPU); change `device` to `cuda:0` for GPU inference.

Expected files under `example_outputs/PBMC_368K/`: `example_roster.json` (96 IDs), `train.h5ad` and `test.h5ad` (48 cells each), and `train.npz` and `test.npz` with `embeddings` of shape `(48, 1024)`. Completion means all 96 embedding `valid` flags are true. This is a small example, not a paper benchmark score.

In [1]:
from pathlib import Path
import json
import os
import sys
import subprocess
import anndata as ad
import numpy as np
from benchmarks.shared._common import roster, embeddings

repo = Path.cwd().resolve()
assert (repo / "benchmarks/embeddings/extract_embeddings.py").is_file(), "Start Jupyter from the repository root"
data_root = Path(os.environ.get("METACELLFORMER_EVALUATION", "../MetacellFormer-evaluation")).expanduser().resolve()
model_dir = Path(os.environ.get("METACELLFORMER_MODEL", "../MetacellFormer-model")).expanduser().resolve()
source = data_root / "inputs/cell_annotation/PBMC_368K.h5ad"
source_roster = data_root / "contracts/cell_type_annotation/PBMC_368K_roster.json"
assert source.is_file() and source_roster.is_file(), "Download the evaluation dataset"
assert (model_dir / "model.safetensors").is_file(), "Download the MetacellFormer model"
work = repo / "example_outputs/PBMC_368K"
work.mkdir(parents=True, exist_ok=True)
config = work / "runtime.json"
config.write_text(json.dumps({"MetacellFormer": {"checkpoint": str(model_dir)}}, indent=2) + "\n")
device = "cpu"  # Change to cuda:0 if available.
full = roster(source_roster)
frame = full.groupby(["split", "label"], sort=False).head(6).reset_index(drop=True)
assert len(frame) == 96 and frame.groupby(["split", "label"]).size().eq(6).all()
roster_file = work / "example_roster.json"
roster_file.write_text(json.dumps({"obs_name": frame.cell_id.tolist(),
                                   "cell_type": frame.label.tolist(),
                                   "official_split": frame.split.tolist()}, indent=2) + "\n")
print("Using", len(frame), "fixed PBMC_368K roster cells (6 per class and split)")

Using 96 fixed PBMC_368K roster cells (6 per class and split)


In [2]:
backed = ad.read_h5ad(source, backed="r")
try:
    for split, name in [("0", "train"), ("2", "test")]:
        ids = frame.loc[frame.split.eq(split), "cell_id"].astype(str).to_numpy()
        positions = backed.obs_names.get_indexer(ids)
        assert len(ids) and (positions >= 0).all()
        part = backed[positions].to_memory()
        assert np.array_equal(part.obs_names.astype(str), ids)
        part.write_h5ad(work / f"{name}.h5ad")
finally:
    backed.file.close()

In [3]:
for name in ("train", "test"):
    subprocess.run([sys.executable, "-m", "benchmarks.embeddings.extract_embeddings",
                    "--input", str(work / f"{name}.h5ad"),
                    "--output", str(work / f"{name}.npz"),
                    "--config", str(config), "--model", "MetacellFormer",
                    "--task", "cell", "--device", device, "--batch-size", "4"],
                   cwd=repo, check=True)
for split, name in [("0", "train"), ("2", "test")]:
    ids = frame.loc[frame.split.eq(split), "cell_id"].astype(str).to_numpy()
    values, returned, valid = embeddings(work / f"{name}.npz", ids)
    assert values.shape[0] == len(ids) and valid.all()
    print(name, values.shape, "valid", int(valid.sum()))

train (48, 1024) valid 48
test (48, 1024) valid 48
